# Speech-to-text learning project

This is a tutorial-guided experiment using a pretrained Wav2Vec2 model. It loads a local audio file, converts it to 16 kHz, runs inference, and decodes the predicted tokens into text. I used this project to understand the steps of a speech-recognition pipeline. It is not a trained model or a production service.


## 1. Load the tools and model

`librosa` loads and resamples audio. Hugging Face supplies the processor and pretrained model; PyTorch runs inference.


In [ ]:
import librosa
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

model_name = "facebook/wav2vec2-base-960h"
processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2ForCTC.from_pretrained(model_name)
model.eval()


## 2. Prepare local audio

Place an audio file named `sample_audio.wav` beside this notebook. Audio is intentionally excluded from the repository for privacy. The model expects 16,000 samples per second.


In [ ]:
audio_file_path = "sample_audio.wav"
audio, sampling_rate = librosa.load(audio_file_path, sr=16000)
input_values = processor(audio, return_tensors="pt", sampling_rate=sampling_rate).input_values
print(f"Loaded {len(audio)} samples at {sampling_rate} Hz")


## 3. Run inference and decode

The model returns logits, or raw scores. The most likely token at each step is selected and decoded into the transcript.


In [ ]:
with torch.no_grad():
    logits = model(input_values).logits

predicted_ids = torch.argmax(logits, dim=-1)
transcription = processor.decode(predicted_ids[0])
print(transcription)
